In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Load data
train_data = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')

train_data['Age'].fillna(train_data['Age'].median(), inplace=True)
train_data['Fare'].fillna(train_data['Fare'].median(), inplace=True)
train_data["Embarked"].describe()

In [ ]:
test_data = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

test_data['Age'].fillna(test_data['Age'].median(), inplace=True)
test_data['Fare'].fillna(test_data['Fare'].median(), inplace=True)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import sklearn.metrics as metrics
import pandas as pd

y = train_data["Survived"]
features = ["Pclass", "Sex", "SibSp", "Parch", "Age", "Embarked"] 
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

X["Child"] = (train_data["Age"] < 13).astype(int)
X_test["Child"] = (test_data["Age"] < 13).astype(int)

X["Young_Adult"] = ((train_data["Age"] >= 18) & (train_data["Age"] < 30)).astype(int)
X_test['Young_Adult'] = ((test_data["Age"] >= 18) & (test_data["Age"] < 30)).astype(int)

X["Adult"] = ((train_data["Age"] >= 30) & (train_data["Age"] < 55)).astype(int)
X_test['Adult'] = ((test_data["Age"] >= 30) & (test_data["Age"] < 55)).astype(int)

X["Elder"] = (train_data["Age"] >= 55).astype(int)
X_test['Elder'] = (test_data["Age"] >= 55).astype(int)

X["Family_Size"] = train_data["SibSp"] + train_data["Parch"]
X_test["Family_Size"] = test_data["SibSp"] + test_data["Parch"]

X["Is_Alone"] = (X["Family_Size"] == 0).astype(int)
X_test["Is_Alone"] = (X_test["Family_Size"] == 0).astype(int)

model = RandomForestClassifier(
            n_estimators=100, 
            max_depth=5,
            class_weight='balanced_subsample',
            random_state=42,
            n_jobs=-1
)

model.fit(X, y)
train_predictions = model.predict(X)

f1 = cross_val_score(model, X, y, cv=5, scoring='f1')
acc = cross_val_score(model, X, y, cv=5, scoring='accuracy')

cm = metrics.confusion_matrix(y, train_predictions)
disp = metrics.ConfusionMatrixDisplay(cm)
disp.plot()

print(f"F1 Scores: {f1}")
print(f"F1 Mean: {f1.mean()}")
print(f"Accuracy Scores: {acc}")
print(f"Accuracy Mean: {acc.mean()}")

test_predictions = model.predict(X_test)

In [ ]:
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': test_predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")